# SECOM Dataset Analysis (Jupyter Notebook Style)

---

## 1. Импорт библиотек

```python
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.decomposition import PCA
from sklearn.feature_selection import VarianceThreshold

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

from sklearn.metrics import classification_report, roc_auc_score, f1_score
```

---

## 2. Загрузка данных

```python
X = pd.read_csv('secom.data', sep=' ', header=None)
y = pd.read_csv('secom_labels.data', sep=' ', header=None)[0]

y = y.replace(-1, 0)

print(X.shape)
print(y.value_counts())
```

**Комментарий:**
Данные содержат большое количество признаков (~590) и бинарную целевую переменную.

---

## 3. Очистка данных

### 3.1 Удаление признаков с пропусками

```python
missing_ratio = X.isna().mean()
X = X.loc[:, missing_ratio < 0.8]
print(X.shape)
```

**Комментарий:**
Удаляем признаки с более чем 80% пропусков, так как они неинформативны.

---

### 3.2 Удаление константных признаков

```python
selector = VarianceThreshold(threshold=0.0)
X = selector.fit_transform(X)
print(X.shape)
```

**Комментарий:**
Константные признаки не влияют на модель.

---

## 4. Разделение данных

```python
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
```

---

## 5. Предобработка и Pipeline

```python
common_steps = [
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=0.95))
]
```

**Комментарий:**
- Заполнение пропусков медианой
- Масштабирование данных
- PCA для уменьшения размерности

---

## 6. Обучение моделей

```python
models = {
    'logreg': (
        LogisticRegression(max_iter=1000),
        {
            'clf__C': [0.01, 0.1, 1, 10]
        }
    ),
    'rf': (
        RandomForestClassifier(),
        {
            'clf__n_estimators': [100, 200],
            'clf__max_depth': [5, 10, 20]
        }
    ),
    'svm': (
        SVC(probability=True),
        {
            'clf__C': [0.1, 1, 10]
        }
    ),
    'knn': (
        KNeighborsClassifier(),
        {
            'clf__n_neighbors': [3, 5, 7]
        }
    )
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
best_models = {}

for name, (model, params) in models.items():
    print(f"\nОбучение модели: {name}")
    
    pipeline = Pipeline(common_steps + [('clf', model)])
    
    grid = GridSearchCV(
        pipeline,
        param_grid=params,
        cv=cv,
        scoring='f1',
        n_jobs=-1
    )
    
    grid.fit(X_train, y_train)
    
    best_models[name] = grid.best_estimator_
    print("Лучшие параметры:", grid.best_params_)
```

---

## 7. Оценка моделей

```python
for name, model in best_models.items():
    print(f"\nМодель: {name}")
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    print(classification_report(y_test, y_pred))
    print("F1:", f1_score(y_test, y_pred))
    print("ROC-AUC:", roc_auc_score(y_test, y_proba))
```

---

## 8. Важность признаков (Random Forest)

```python
if 'rf' in best_models:
    rf_model = best_models['rf'].named_steps['clf']
    importances = rf_model.feature_importances_
    
    indices = np.argsort(importances)[-10:][::-1]
    
    plt.figure()
    plt.title("Top 10 Features")
    plt.bar(range(len(indices)), importances[indices])
    plt.show()
```

---

## 9. Вывод

- Проведена очистка данных
- Применены методы снижения размерности
- Обучены несколько моделей
- Выбрана лучшая модель по F1-score

Модель может быть использована для прогнозирования дефектов в производстве.
